# SAM3 Hugging Face - all checkpoints

This notebook runs all six checkpoints and skips easy_semantic, medium_instance, and hard_panoptic.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
HF_DIR = Path('/content/drive/MyDrive/huggingface')
os.environ['HF_HOME'] = str(HF_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_DIR / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(HF_DIR / 'transformers')
HF_DIR.mkdir(parents=True, exist_ok=True)
print('Cache:', HF_DIR)

In [ ]:
!pip install -q -U transformers accelerate huggingface_hub pycocotools safetensors

In [ ]:
from huggingface_hub import login
login()
# The account must have access to facebook/sam3.

In [ ]:
# Keep False after a successful download. Set True only for a corrupted cache.
RESET_SAM3_CACHE = False

import shutil
from huggingface_hub import snapshot_download
model_cache = HF_DIR / 'hub' / 'models--facebook--sam3'
if RESET_SAM3_CACHE and model_cache.exists():
    shutil.rmtree(model_cache)
    print('Removed old SAM3 cache.')
snapshot_download(repo_id='facebook/sam3', cache_dir=str(HF_DIR / 'hub'), resume_download=True)
print('SAM3 cache is ready:', model_cache)

In [ ]:
REPO_URL = 'https://github.com/<username>/<repo>.git'
REPO_DIR = '/content/image-segmentation'
import subprocess
subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR

In [ ]:
CHECKPOINT_TASKS = ['cp1_holes', 'cp2_slice', 'cp3_thin', 'cp4_curb', 'cp5_occlusion', 'cp6_coverage']
TASK_DIRS = {
    'cp1_holes': 'data/checkpoints/cp1_holes',
    'cp2_slice': 'data/checkpoints/cp2_slice',
    'cp3_thin': 'data/checkpoints/cp3_thin',
    'cp4_curb': 'data/checkpoints/cp4_curb',
    'cp5_occlusion': 'data/checkpoints/cp5_occlusion',
    'cp6_coverage': 'data/checkpoints/cp6_coverage',
}
print('Tasks to run:', ', '.join(CHECKPOINT_TASKS))

In [ ]:
import subprocess
from pathlib import Path
for task in CHECKPOINT_TASKS:
    task_dir = Path(TASK_DIRS[task])
    images = task_dir / 'images'
    classes = task_dir / 'classes.json'
    output = Path('submissions') / f'{task}.zip'
    cmd = ['python', 'scripts/sam3_hf_coco.py', '--images', str(images), '--classes', str(classes), '--output', str(output), '--device', 'cuda', '--threshold', '0.25', '--format', 'auto']
    print(f'\n=== Running {task} ===')
    subprocess.run(cmd, check=True)
    print('Created:', output)

In [ ]:
import subprocess
for task in CHECKPOINT_TASKS:
    print(f'\n=== Checking {task} ===')
    subprocess.run(['python', 'scripts/inspect_submissions.py', '--task', task, '--dir', 'submissions'], check=True)

In [ ]:
from google.colab import files
for task in CHECKPOINT_TASKS:
    files.download(f'submissions/{task}.zip')